In [1]:
import psycopg2
from datetime import datetime

conn = psycopg2.connect(
    host="localhost",
    database="kenexaihackathon",
    user="postgres",
    password="09092002"
)

cursor = conn.cursor()

# -----------------------------------
# LOAD DIMENSION TABLES
# -----------------------------------

def load_devices():

    cursor.execute("""
        SELECT DISTINCT device_name, device_identifier, source_system
        FROM silver.alerts_clean
    """)

    rows = cursor.fetchall()

    for r in rows:

        cursor.execute("""
        INSERT INTO gold.dim_device (device_name, device_identifier, source_system)
        VALUES (%s,%s,%s)
        ON CONFLICT DO NOTHING
        """, r)


def load_organizations():

    cursor.execute("""
        SELECT DISTINCT organization_name
        FROM silver.alerts_clean
    """)

    rows = cursor.fetchall()

    for r in rows:

        cursor.execute("""
        INSERT INTO gold.dim_organization (organization_name)
        VALUES (%s)
        ON CONFLICT DO NOTHING
        """, r)


def load_alert_types():

    cursor.execute("""
        SELECT DISTINCT alert_type, service_name
        FROM silver.alerts_clean
    """)

    rows = cursor.fetchall()

    for r in rows:

        cursor.execute("""
        INSERT INTO gold.dim_alert_type (alert_type, service_name)
        VALUES (%s,%s)
        ON CONFLICT DO NOTHING
        """, r)


# -----------------------------------
# TIME DIMENSION
# -----------------------------------

def get_time_key(ts):

    cursor.execute("""
        SELECT time_key
        FROM gold.dim_time
        WHERE event_time = %s
    """, (ts,))

    result = cursor.fetchone()

    if result:
        return result[0]

    cursor.execute("""
        INSERT INTO gold.dim_time
        (event_time, event_date, hour, day, month, year)
        VALUES (%s,%s,%s,%s,%s,%s)
        RETURNING time_key
    """,
    (
        ts,
        ts.date(),
        ts.hour,
        ts.day,
        ts.month,
        ts.year
    ))

    return cursor.fetchone()[0]


# -----------------------------------
# LOAD FACT INCIDENTS
# -----------------------------------

def load_incidents():

    cursor.execute("""
        SELECT
            correlation_id,
            source_system,
            organization_name,
            device_name,
            device_identifier,
            alert_type,
            severity,
            MIN(event_time),
            MAX(event_time),
            COUNT(*),
            BOOL_OR(synthetic)

        FROM silver.alerts_clean

        GROUP BY
        correlation_id,
        source_system,
        organization_name,
        device_name,
        device_identifier,
        alert_type,
        severity
    """)

    rows = cursor.fetchall()

    for r in rows:

        correlation_id = r[0]
        source = r[1]
        org = r[2]
        device = r[3]
        device_identifier = r[4]
        alert_type = r[5]
        severity = r[6]
        start_time = r[7]
        end_time = r[8]
        alert_count = r[9]
        synthetic = r[10]

        # get dimension keys

        cursor.execute("""
        SELECT device_key
        FROM gold.dim_device
        WHERE device_name=%s
        """,(device,))

        device_key = cursor.fetchone()[0]

        cursor.execute("""
        SELECT organization_key
        FROM gold.dim_organization
        WHERE organization_name=%s
        """,(org,))

        org_key = cursor.fetchone()[0]

        cursor.execute("""
        SELECT alert_type_key
        FROM gold.dim_alert_type
        WHERE alert_type=%s
        """,(alert_type,))

        alert_key = cursor.fetchone()[0]

        start_time_key = get_time_key(start_time)
        end_time_key = get_time_key(end_time)

        duration = (end_time - start_time) if end_time and start_time else None

        cursor.execute("""
        INSERT INTO gold.fact_incidents
        (
        correlation_id,
        device_key,
        organization_key,
        alert_type_key,
        start_time_key,
        end_time_key,
        incident_duration,
        alert_count,
        severity,
        synthetic
        )
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
        """,
        (
        correlation_id,
        device_key,
        org_key,
        alert_key,
        start_time_key,
        end_time_key,
        duration,
        alert_count,
        severity,
        synthetic
        ))


# -----------------------------------
# RUN PIPELINE
# -----------------------------------

print("Loading dimension tables...")

load_devices()
load_organizations()
load_alert_types()

print("Loading incidents...")

load_incidents()

conn.commit()

cursor.close()
conn.close()

print("Gold layer ETL completed.")


Loading dimension tables...
Loading incidents...
Gold layer ETL completed.
